##**Import Libraries and Define Custom Functions**

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import BaggingRegressor # NEW
from sklearn import set_config
set_config(transform_output='pandas')

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
def regression_metrics(y_true, y_pred, label='', verbose = True, output_dict=False):
  # Get metrics
  mae = mean_absolute_error(y_true, y_pred)
  mse = mean_squared_error(y_true, y_pred)
  rmse = mean_squared_error(y_true, y_pred)
  r_squared = r2_score(y_true, y_pred)
  if verbose == True:
    # Print Result with Label and Header
    header = "-"*60
    print(header, f"Regression Metrics: {label}", header, sep='\n')
    print(f"- MAE = {mae:,.3f}")
    print(f"- MSE = {mse:,.3f}")
    print(f"- RMSE = {rmse:,.3f}")
    print(f"- R^2 = {r_squared:,.3f}")
  if output_dict == True:
      metrics = {'Label':label, 'MAE':mae,
                 'MSE':mse, 'RMSE':rmse, 'R^2':r_squared}
      return metrics
def evaluate_regression(reg, X_train, y_train, X_test, y_test, verbose = True,
                        output_frame=False):
  # Get predictions for training data
  y_train_pred = reg.predict(X_train)
  # Call the helper function to obtain regression metrics for training data
  results_train = regression_metrics(y_train, y_train_pred, verbose = verbose,
                                     output_dict=output_frame,
                                     label='Training Data')
  print()
  # Get predictions for test data
  y_test_pred = reg.predict(X_test)
  # Call the helper function to obtain regression metrics for test data
  results_test = regression_metrics(y_test, y_test_pred, verbose = verbose,
                                  output_dict=output_frame,
                                    label='Test Data' )
  # Store results in a dataframe if ouput_frame is True
  if output_frame:
    results_df = pd.DataFrame([results_train,results_test])
    # Set the label as the index
    results_df = results_df.set_index('Label')
    # Set index.name to none to get a cleaner looking result
    results_df.index.name=None
    # Return the dataframe
    return results_df.round(3)

##**Load the Dataset**

In [ ]:
# Mount google drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
# Using the recommended file structure.
fpath = "/content/drive/MyDrive/AXSOSACADEMY/02-MachineLearning/Week06/auto_mpg_dirty.csv"
df = pd.read_csv(fpath)
df.info()
df.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 398 entries, 0 to 397
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   mpg           398 non-null    float64
 1   cylinders     398 non-null    int64  
 2   displacement  398 non-null    float64
 3   horsepower    392 non-null    float64
 4   weight        355 non-null    float64
 5   acceleration  375 non-null    float64
 6   model year    398 non-null    int64  
 7   origin        333 non-null    object 
 8   car name      398 non-null    object 
dtypes: float64(5), int64(2), object(2)
memory usage: 28.1+ KB


,mpg,cylinders,displacement,horsepower,weight,acceleration,model year,origin,car name
0,18.0,8,307.0,130.0,3504.0,12.0,70,America,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693.0,11.5,70,America,buick skylark 320
2,18.0,8,318.0,150.0,3436.0,11.0,70,America,plymouth satellite
3,16.0,8,304.0,150.0,3433.0,12.0,70,America,amc rebel sst
4,17.0,8,302.0,140.0,3449.0,10.5,70,America,ford torino


##**Create Preprocessing Object**

**Define X and y and Train Test Split**

In [ ]:
# Arrange Data into Features Matrix and Target Vector
y = df['mpg']
X = df.drop(columns = ['mpg','car name'])
# Split the data for validation
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

**Check for Null Values**

In [ ]:
# Check for null values
X_train.isna().sum()

,0
cylinders,0
displacement,0
horsepower,5
weight,30
acceleration,19
model year,0
origin,47


**Check for Inconsistent Values in Categorical Columns**

In [ ]:
# Check for inconsitent values in categorical columns
X_train['origin'].value_counts()

,count
origin,
America,163
Asia,48
Europe,40


**Preprocessing Pipeline for Categorical Features**

In [ ]:
# Saving list of categorical columns
cat_cols = X_train.select_dtypes('object').columns
# Constructing categorical preprocessing objects
cat_imputer = SimpleImputer(strategy='constant', fill_value='MISSING')
ohe_encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
cat_pipe = make_pipeline(cat_imputer,ohe_encoder)
cat_tuple = ('cat',cat_pipe, cat_cols)
cat_tuple

('cat',
 Pipeline(steps=[('simpleimputer',
                  SimpleImputer(fill_value='MISSING', strategy='constant')),
                 ('onehotencoder',
                  OneHotEncoder(handle_unknown='ignore', sparse_output=False))]),
 Index(['origin'], dtype='object'))

**Preprocessing Pipeline for Numerical Features**

In [ ]:
# Save list of numeric columns
num_cols = X_train.select_dtypes('number').columns
# Constructing numeric preprocesssing objects
num_imputer = SimpleImputer(strategy='median')
scaler = StandardScaler()
num_pipe = make_pipeline(num_imputer, scaler)
num_tuple = ('num',num_pipe, num_cols)
num_tuple

('num',
 Pipeline(steps=[('simpleimputer', SimpleImputer(strategy='median')),
                 ('standardscaler', StandardScaler())]),
 Index(['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration',
        'model year'],
       dtype='object'))

**Create Column Transformer**

In [ ]:
# Define a column transformer
preprocessor  = ColumnTransformer([num_tuple, cat_tuple],
                                  verbose_feature_names_out=False)

**Train and Evaluate the Default Model (Using a model pipeline)**

In [ ]:
# Instantiate a Default Model
bagreg = BaggingRegressor(random_state = 42)
# Model Pipeline with default preprocessor and default model
bagreg_pipe = make_pipeline(preprocessor, bagreg)
# Fit the model pipeline on the training data only
bagreg_pipe.fit(X_train, y_train)
# Call custom function for evaluation
evaluate_regression(bagreg_pipe, X_train, y_train, X_test, y_test)

------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 0.907
- MSE = 1.946
- RMSE = 1.946
- R^2 = 0.969

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 1.939
- MSE = 7.590
- RMSE = 7.590
- R^2 = 0.866


**Tune the Model with GridSearchCV**

In [ ]:
# Obtain list of parameters
bagreg_pipe.get_params()

{'memory': None,
 'steps': [('columntransformer',
   ColumnTransformer(transformers=[('num',
                                    Pipeline(steps=[('simpleimputer',
                                                     SimpleImputer(strategy='median')),
                                                    ('standardscaler',
                                                     StandardScaler())]),
                                    Index(['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration',
          'model year'],
         dtype='object')),
                                   ('cat',
                                    Pipeline(steps=[('simpleimputer',
                                                     SimpleImputer(fill_value='MISSING',
                                                                   strategy='constant')),
                                                    ('onehotencoder',
                                                     OneHotEncoder(handle_unkn

In [ ]:
# Define parameters to tune
param_grid = {'baggingregressor__n_estimators': [5, 10, 20, 30, 40, 50],
              'baggingregressor__max_samples' : [.5, .7, .9, ],
              'baggingregressor__max_features': [.5, .7, .9 ]}
# Instaniate the gridsearch
gridsearch = GridSearchCV(bagreg_pipe, param_grid, n_jobs=-1, verbose=1)
# Fit the gridsearch on the training data
gridsearch.fit(X_train, y_train)
# Obtain the best paramters from the gridsearch
gridsearch.best_params_

Fitting 5 folds for each of 54 candidates, totalling 270 fits


{'baggingregressor__max_features': 0.9,
 'baggingregressor__max_samples': 0.7,
 'baggingregressor__n_estimators': 40}

In [ ]:
# Define a model with the best parameters already refit on the entire training set
best_bagreg_grid = gridsearch.best_estimator_
# Evalute the tuned model
evaluate_regression(best_bagreg_grid, X_train, y_train, X_test, y_test)

------------------------------------------------------------
Regression Metrics: Training Data
------------------------------------------------------------
- MAE = 1.040
- MSE = 2.527
- RMSE = 2.527
- R^2 = 0.959

------------------------------------------------------------
Regression Metrics: Test Data
------------------------------------------------------------
- MAE = 1.809
- MSE = 6.336
- RMSE = 6.336
- R^2 = 0.888
